# ajustando as colunas null

In [ ]:
cursor.execute("""
CREATE TABLE proposicao_autores (
    id INT AUTO_INCREMENT PRIMARY KEY,
    fk_proposicao INT,
    fk_deputado INT,
    FOREIGN KEY (fk_proposicao) REFERENCES proposicoes(cd_proposicoes),
    FOREIGN KEY (fk_deputado) REFERENCES deputado(cd_deputado)
);
""")
conn.commit()
print("Tabela 'proposicao_autores' criada com sucesso!")

Tabela 'proposicao_autores' criada com sucesso!


Criando

In [ ]:
cursor = conn.cursor()


sql_create = """
CREATE TABLE IF NOT EXISTS proposicao_tema (
    id_proposicao INT,
    id_tema INT,
    PRIMARY KEY (id_proposicao, id_tema)
);
"""

try:
    cursor.execute(sql_create)
    conn.commit()
    print("✅ Tabela 'proposicao_tema' criada ou já existente!")
except mysql.connector.Error as err:
    print(f"❌ Erro ao criar tabela: {err}")

✅ Tabela 'proposicao_tema' criada ou já existente!


In [ ]:
#preenchendo nova tabela
sql_insert = """
INSERT IGNORE INTO proposicao_tema (id_proposicao, id_tema)
SELECT cd_proposicoes, fk_tema FROM proposicoes
WHERE fk_tema IS NOT NULL;
"""

try:
    cursor.execute(sql_insert)
    conn.commit()
    print(f"✅ Sucesso! {cursor.rowcount} registros migrados usando 'cd_proposicoes'.")
except mysql.connector.Error as err:
    print(f"❌ Erro na migração: {err}")

✅ Sucesso! 46893 registros migrados usando 'cd_proposicoes'.


In [ ]:
import pandas as pd

# Filtra e prepara os dados
df_limpo = df_autores_validos.dropna(subset=['idProposicao', 'idDeputadoAutor'])

# Criamos a lista para INSERIR (e não atualizar)
# Ordem: (id_da_proposição, id_do_deputado)
dados_para_inserir = [
    (int(prop), int(dep))
    for prop, dep in zip(df_limpo['idProposicao'], df_limpo['idDeputadoAutor'])
]

# Comando de INSERÇÃO
sql_insert = "INSERT INTO proposicao_autores (fk_proposicao, fk_deputado) VALUES (%s, %s)"

for i in range(0, len(dados_para_inserir), 1000):
    lote = dados_para_inserir[i:i+1000]
    try:
        cursor.executemany(sql_insert, lote)
        conn.commit()
        print(f"Inseridos {i + len(lote)} vínculos de autoria...")
    except Exception as e:
        print(f"Erro: {e}")
        conn.rollback()

print("Concluído! Agora todos os autores estão vinculados.")